Building an End-to-End Data Engineering Pipeline for E-Commerce Order Analytics

Step 1: EXTRACT - Baca Data Mentah

In [56]:
from pathlib import Path
import pandas as pd
import numpy as np  
import ast
# Import semua data raw 
data = pd.read_csv("../data/raw/raw_payments.csv")

data.head(20)

,payment_id,order_id,metode_pembayaran,status_pembayaran,jumlah_bayar,biaya_admin,tanggal_bayar
0,PAY-5001,ORD-10074,virtual account,refunded,42000.0,0.0,2024-07-23 00:13:00
1,PAY-5002,ORD-10123,E-Wallet,Pending,45000.0,NaN,2024-07-02 09:08:00
2,PAY-5003,ORD-10043,cod,Paid,25000.0,0.0,17/05/2024
3,PAY-5004,ORD-10103,bank transfer,Paid,38000.0,NaN,2024-06-02 04:08:00
4,PAY-5005,ORD-10072,CREDIT_CARD,paid,34000.0,0.0,2024-06-27
5,PAY-5006,ORD-10110,credit card,Pending,42000.0,0.0,"Jul 08, 2024"
6,PAY-5007,ORD-10117,e-wallet,failed,149000.0,0.0,2024-05-24 06:45:00
7,PAY-5008,ORD-10004,virtual account,Paid,149000.0,2500.0,2024-05-02 14:55:00
8,PAY-5009,ORD-10075,credit card,failed,34000.0,5000.0,"Jun 02, 2024"
9,PAY-5010,ORD-10021,virtual account,paid,34000.0,0.0,2024-06-07 18:06:00


In [57]:
# Inspeksi awal
print(f"Jumlah baris: {len(data)}")
print(f"Kolom: {list(data.columns)}")
data.info()

Jumlah baris: 62
Kolom: ['payment_id', 'order_id', 'metode_pembayaran', 'status_pembayaran', 'jumlah_bayar', 'biaya_admin', 'tanggal_bayar']
<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   payment_id         62 non-null     str    
 1   order_id           62 non-null     str    
 2   metode_pembayaran  62 non-null     str    
 3   status_pembayaran  62 non-null     str    
 4   jumlah_bayar       57 non-null     float64
 5   biaya_admin        55 non-null     float64
 6   tanggal_bayar      62 non-null     str    
dtypes: float64(2), str(5)
memory usage: 3.5 KB


In [58]:
print("\nDuplikasi")
print(f"{data.duplicated().sum()}")


Duplikasi
0


In [59]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
payment_id           0
order_id             0
metode_pembayaran    0
status_pembayaran    0
jumlah_bayar         5
biaya_admin          7
tanggal_bayar        0
dtype: int64


In [60]:
data.nunique()

payment_id           62
order_id             61
metode_pembayaran    12
status_pembayaran     7
jumlah_bayar         10
biaya_admin           4
tanggal_bayar        57
dtype: int64

In [61]:
# Nilai Negatif
refund_negatif = data[data['biaya_admin'] < 0]
print(refund_negatif)

Empty DataFrame
Columns: [payment_id, order_id, metode_pembayaran, status_pembayaran, jumlah_bayar, biaya_admin, tanggal_bayar]
Index: []


Step 2: TRANSFORM - Bersihkan Data

In [62]:
# Tipe diskon: Semua huruf kecil
data['metode_pembayaran'] = data['metode_pembayaran'].str.strip().str.replace('_', ' ', regex=False).str.title()

# Status: Huruf pertama kapital
data['status_pembayaran'] = data['status_pembayaran'].str.strip().str.capitalize()

print(data[['metode_pembayaran', 'status_pembayaran']])

   metode_pembayaran status_pembayaran
0    Virtual Account          Refunded
1           E-Wallet           Pending
2                Cod              Paid
3      Bank Transfer              Paid
4        Credit Card              Paid
..               ...               ...
57     Bank Transfer              Paid
58   Virtual Account              Paid
59          E-Wallet          Refunded
60               Cod              Paid
61              Qris              Paid

[62 rows x 2 columns]


In [63]:
# Mengisi missing value pada biaya_admin menjadi 0.0
data['biaya_admin'] = data['biaya_admin'].fillna(0.0)
print("Missing value biaya_admin:")
print(data['biaya_admin'].isna().sum())
# Menghapus baris yang memiliki missing value pada jumlah_bayar
data = data.dropna(subset=['jumlah_bayar'])
print("\nMissing value jumlah_bayar:")
print(data['jumlah_bayar'].isna().sum())

Missing value biaya_admin:
0

Missing value jumlah_bayar:
0


In [64]:
data.head()

,payment_id,order_id,metode_pembayaran,status_pembayaran,jumlah_bayar,biaya_admin,tanggal_bayar
0,PAY-5001,ORD-10074,Virtual Account,Refunded,42000.0,0.0,2024-07-23 00:13:00
1,PAY-5002,ORD-10123,E-Wallet,Pending,45000.0,0.0,2024-07-02 09:08:00
2,PAY-5003,ORD-10043,Cod,Paid,25000.0,0.0,17/05/2024
3,PAY-5004,ORD-10103,Bank Transfer,Paid,38000.0,0.0,2024-06-02 04:08:00
4,PAY-5005,ORD-10072,Credit Card,Paid,34000.0,0.0,2024-06-27


In [65]:
# Mengubah jumlah_bayar', biaya_admin menjadi nilai angka normal
(data[['jumlah_bayar', 'biaya_admin']].isna().sum())
data['jumlah_bayar'] = data['jumlah_bayar'].astype(int)
data['biaya_admin'] = data['biaya_admin'].astype(int)
data.head(10)

,payment_id,order_id,metode_pembayaran,status_pembayaran,jumlah_bayar,biaya_admin,tanggal_bayar
0,PAY-5001,ORD-10074,Virtual Account,Refunded,42000,0,2024-07-23 00:13:00
1,PAY-5002,ORD-10123,E-Wallet,Pending,45000,0,2024-07-02 09:08:00
2,PAY-5003,ORD-10043,Cod,Paid,25000,0,17/05/2024
3,PAY-5004,ORD-10103,Bank Transfer,Paid,38000,0,2024-06-02 04:08:00
4,PAY-5005,ORD-10072,Credit Card,Paid,34000,0,2024-06-27
5,PAY-5006,ORD-10110,Credit Card,Pending,42000,0,"Jul 08, 2024"
6,PAY-5007,ORD-10117,E-Wallet,Failed,149000,0,2024-05-24 06:45:00
7,PAY-5008,ORD-10004,Virtual Account,Paid,149000,2500,2024-05-02 14:55:00
8,PAY-5009,ORD-10075,Credit Card,Failed,34000,5000,"Jun 02, 2024"
9,PAY-5010,ORD-10021,Virtual Account,Paid,34000,0,2024-06-07 18:06:00


In [66]:
# Mengubah format tanggal menjadi standar ISO 2024-06-27
# Mengubah berbagai format tanggal menjadi datetime
data['tanggal_bayar'] = pd.to_datetime(
    data['tanggal_bayar'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)
data['tanggal_bayar'] = data['tanggal_bayar'].dt.strftime('%Y-%m-%d')
print(data['tanggal_bayar'].isna().sum())
data.head()

0


,payment_id,order_id,metode_pembayaran,status_pembayaran,jumlah_bayar,biaya_admin,tanggal_bayar
0,PAY-5001,ORD-10074,Virtual Account,Refunded,42000,0,2024-07-23
1,PAY-5002,ORD-10123,E-Wallet,Pending,45000,0,2024-02-07
2,PAY-5003,ORD-10043,Cod,Paid,25000,0,2024-05-17
3,PAY-5004,ORD-10103,Bank Transfer,Paid,38000,0,2024-02-06
4,PAY-5005,ORD-10072,Credit Card,Paid,34000,0,2024-06-27


In [67]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
payment_id           0
order_id             0
metode_pembayaran    0
status_pembayaran    0
jumlah_bayar         0
biaya_admin          0
tanggal_bayar        0
dtype: int64


In [68]:
# Simpan ke CSV data setelah proses atau clean
data.to_csv(
    "../data/warehouse/payments_clean.csv",
    index=False,
    encoding="utf-8")
# Melihat kembali data yang disimpan
data = pd.read_csv("../data/warehouse/payments_clean.csv")
data.head()

,payment_id,order_id,metode_pembayaran,status_pembayaran,jumlah_bayar,biaya_admin,tanggal_bayar
0,PAY-5001,ORD-10074,Virtual Account,Refunded,42000,0,2024-07-23
1,PAY-5002,ORD-10123,E-Wallet,Pending,45000,0,2024-02-07
2,PAY-5003,ORD-10043,Cod,Paid,25000,0,2024-05-17
3,PAY-5004,ORD-10103,Bank Transfer,Paid,38000,0,2024-02-06
4,PAY-5005,ORD-10072,Credit Card,Paid,34000,0,2024-06-27
